# E19s: OMNI-DC fairness ablation (fit-free A-dual arms), seeds 0-9

Adds two arms to E19c that fit nothing: **A-dual-const** (shift 0, scale 1) and **A-dual-med** (shift = median residual of the image's structured anchors, scale 1). Both are valid by Theorem 1 and remove the distance model whose scale collapses on OMNI-DC. CPU only; about 10-20 min.

**Before you start**
1. Upload `CV_Project/experiments/e19_omnidc.py` (the updated one) to `MyDrive/spacal/e19_omnidc.py`.
2. `MyDrive/spacal/omni_code.tgz`, `omnidc_merged/` and `prepared_new/arkitscenes__unidepthv2-large.npz` are already there from E19c.
3. Runtime: CPU is enough (High-RAM if available). Run all cells; the last one downloads `e19s_results.tgz`. **Stop the runtime afterwards.**

In [ ]:
# 1. Drive and inputs
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, glob, time
D = '/content/drive/MyDrive/spacal'
for f in ('omni_code.tgz', 'e19_omnidc.py', 'omnidc_merged', 'prepared_new/arkitscenes__unidepthv2-large.npz'):
    assert os.path.exists(f'{D}/{f}'), f'missing MyDrive/spacal/{f}'
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode: raise RuntimeError(f'failed ({r.returncode}): {cmd[:80]}')

In [ ]:
# 2. Code (bundle + updated E19 script), frozen-model table, merged OMNI-DC predictions (~5 min)
A, MERGED = '/content/omni_anchors/arkitscenes', '/content/omnidc_merged'
sh(f'cd /content && rm -rf CV_Project omni_anchors && tar xzf {D}/omni_code.tgz -C /content')
sh(f'cp {D}/e19_omnidc.py /content/CV_Project/experiments/e19_omnidc.py && grep -c simple-only /content/CV_Project/experiments/e19_omnidc.py')
os.makedirs('/content/prepared', exist_ok=True); os.makedirs(MERGED, exist_ok=True)
sh(f'cp -n {D}/prepared_new/arkitscenes__unidepthv2-large.npz /content/prepared/')
sh(' && '.join(f'cp -n {D}/omnidc_merged/{p}__k200_seed{s}_draw0__clean.npz {MERGED}/' for p in ('uniform', 'cluster') for s in range(10)))
sh(f'ls {MERGED} | wc -l')

In [ ]:
# 3. Fit-free arms for seeds 0-9 (3 at a time). Results go to Drive; finished seeds are skipped.
env = dict(os.environ, PYTHONPATH='/content/CV_Project', MPLCONFIGDIR='/content/mpl', PYTHONUNBUFFERED='1',
           SPACAL_RESULTS_ROOT=f'{D}/results_omnidc', SPACAL_PREPARED_ROOT='/content/prepared', OMP_NUM_THREADS='2')
def done(s): return bool(glob.glob(f'{D}/results_omnidc/e19_simple_s{s}/**/summary.csv', recursive=True))
todo = [s for s in range(10) if not done(s)]
print('seeds to run:', todo)
os.makedirs(f'{D}/logs/e19s', exist_ok=True)
for i in range(0, len(todo), 3):
    batch = todo[i:i + 3]
    ps = {s: subprocess.Popen(['python', '-W', 'ignore', '-m', 'experiments.e19_omnidc', '--exp-id', f'e19_simple_s{s}',
                               '--anchors', A, '--omni', MERGED, '--patterns', 'uniform', 'cluster', '--seeds', str(s),
                               '--draws', '0', '--omni-sources', 'clean', '--simple-only'],
                              cwd='/content/CV_Project', env=env,
                              stdout=open(f'{D}/logs/e19s/e19s_s{s}.log', 'w'), stderr=subprocess.STDOUT) for s in batch}
    for s, p in ps.items():
        p.wait(); print(f'seed {s}: exit {p.returncode}  {time.strftime("%H:%M:%S")}', flush=True)
missing = [s for s in range(10) if not done(s)]
print('ALL DONE' if not missing else f'MISSING seeds {missing}: see MyDrive/spacal/logs/e19s/, then rerun this cell')

In [ ]:
# 4. Archive and download
import tarfile
with tarfile.open(f'{D}/e19s_results.tgz', 'w:gz') as t:
    for f in sorted(glob.glob(f'{D}/results_omnidc/e19_simple_s*/**/*', recursive=True)):
        if f.endswith(('.csv', '.json', '.md')):
            t.add(f, arcname=os.path.relpath(f, f'{D}/results_omnidc'))
print('archive:', os.path.getsize(f'{D}/e19s_results.tgz') // 1024, 'KB')
from google.colab import files
files.download(f'{D}/e19s_results.tgz')